# Lesson 8.3: R-Squared and Residuals: Is the Line Any Good?

Measure how good the study-hours line is with R squared and adjusted R squared, and use the residuals to check the fit, for the lesson's eight students.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Fit the model with statsmodels

We fit the same line as in Lesson 8.2, but with statsmodels' formula interface, which gives the full summary. The formula 'score ~ hours' reads as 'score explained by hours'.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

df = pd.DataFrame({'hours': [1, 2, 3, 4, 5, 6, 7, 8],
                   'score': [48, 62, 55, 66, 63, 74, 70, 82]})

model = smf.ols('score ~ hours', data=df).fit()

print(model.params.round(3))

## 2. R squared and adjusted R squared

R squared is the share of the score variation that the line explains (0 to 1). Adjusted R squared is slightly lower: it charges a small penalty for each predictor, so it is fairer when comparing models.

In [2]:
print(f'R squared          = {model.rsquared:.3f}')
print(f'adjusted R squared = {model.rsquared_adj:.3f}')
print(f'residual std error = {np.sqrt(model.scale):.2f} points')

sse = (model.resid ** 2).sum()
print(f'SSE = {sse:.1f}')

## 3. Look at the residuals

A residual is what the line missed: actual minus predicted. Residuals should add up to about zero, and no student should be missed by a wildly larger amount than the rest.

In [3]:
df['fitted'] = model.fittedvalues
df['residual'] = model.resid

print(df.round(2).to_string(index=False))
print(f'sum of residuals = {df.residual.sum():.6f}')

## 4. Plot residuals against fitted values

If a straight line suits the data, the points scatter randomly around zero with no curve and no funnel shape. A visible pattern would warn us that the model is missing something.

In [4]:
plt.figure(figsize=(6, 3.4))
plt.scatter(df.fitted, df.residual, color='#1f77b4')
plt.axhline(0, color='#d62728', linestyle='--')
plt.xlabel('Fitted score')
plt.ylabel('Residual')
plt.title('Residuals versus fitted values')
plt.show()

## Try this

- Change the last score from 82 to 60. Predict what happens to R squared and to the biggest residual, then run it.
- Look at the residual plot. Which student is missed most, and in which direction?

## The same idea in R, SPSS and Stata

**R**

```r
hours <- c(1, 2, 3, 4, 5, 6, 7, 8)
score <- c(48, 62, 55, 66, 63, 74, 70, 82)
fit <- lm(score ~ hours)
summary(fit)
round(residuals(fit), 2)
plot(fitted(fit), residuals(fit)); abline(h = 0)
```

**SPSS** (Analyze > Regression > Linear (Save: Residuals and Predicted values; Plots: *ZRESID on Y, *ZPRED on X))

```
DATA LIST FREE / hours score.
BEGIN DATA
1 48  2 62  3 55  4 66  5 63  6 74  7 70  8 82
END DATA.
REGRESSION
  /STATISTICS COEFF R ANOVA
  /DEPENDENT score
  /METHOD=ENTER hours
  /SCATTERPLOT=(*ZRESID ,*ZPRED)
  /SAVE PRED RESID.
```

**Stata**

```
clear
set obs 8
generate hours = _n
generate score = real(word("48 62 55 66 63 74 70 82", _n))
regress score hours
predict fitted, xb
predict resid, residuals
list hours score fitted resid
rvfplot, yline(0)
```